## Imports

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import pytz

## Define the methods to:
* Fetch the tickers current prices, EPS and trainling PE
* Get the historic close prices to calculate change

In [2]:
def get_ticker_details(ticker_symbol):
    try:
        ticker_obj = yf.Ticker(ticker_symbol)
        info = ticker_obj.info
        
        # Extracting the requested fields
        price = info.get('currentPrice') or info.get('regularMarketPrice')
        currency = info.get('currency', 'Unknown')
        eps = info.get('trailingEps')  # Earnings Per Share
        pe_ratio = info.get('trailingPE')  # P/E Ratio
        sector = info.get('sector', 'N/A')
        
        return pd.Series([price, currency, eps, pe_ratio, sector])
    
    except Exception as e:
        # Returning None for all columns if the fetch fails
        return pd.Series([None, "Error", None, None, None])

In [3]:
def get_last_prices_dict(ticker_column):
    """
    Takes a pandas Series of tickers and returns a dictionary 
    mapping {Ticker: Last_Close_Price}.
    """
    # 1. Get unique tickers as a list
    unique_tickers = ticker_column.unique().tolist()
    
    # 2. Fetch last 1 day of data
    # We use 'Close' specifically to simplify the returned object
    data = yf.download(unique_tickers, period="1d", progress=False)['Close']
    
    # 3. Handle data structure based on ticker count
    # yfinance returns a Series for 1 ticker and a DataFrame for 2+
    if isinstance(data, pd.DataFrame):
        # Take the last available row and convert to dict
        return data.iloc[-1].to_dict()
    else:
        # It's a Series (single ticker), return dict with the first ticker name
        return {unique_tickers[0]: data.iloc[-1]}

In [4]:
def get_pct_change_dict(ticker_column):
    """
    Fetches the last 2 sessions for a Series of tickers and 
    returns a dictionary mapping {Ticker: Percentage_Change}.
    """
    unique_tickers = ticker_column.unique().tolist()
    
    # Fetch 2 days to get (Today's Close - Yesterday's Close) / Yesterday's Close
    # We use 'Close' only to keep the object lightweight
    data = yf.download(unique_tickers, period="2d", progress=False)['Close']
    
    # Calculate percentage change between the two rows (axis=0 is the default)
    # .pct_change() calculates: (Current - Previous) / Previous
    pct_changes = data.pct_change().iloc[-1] # Grab only the most recent calculation
    
    # Handle single ticker (Series) vs Multiple tickers (DataFrame)
    if isinstance(pct_changes, pd.Series):
        return pct_changes.to_dict()
    else:
        # If only 1 ticker was passed, pct_changes is a float
        return {unique_tickers[0]: pct_changes}

In [20]:
import pandas as pd
import yfinance as yf

def get_complete_stock_analysis(file_path):
    # 1. Load tickers
    df_tickers = pd.read_csv(file_path)
    tickers = df_tickers['Ticker'].unique().tolist()

    # 2. Batch download 1 year of data (Fastest for all technicals)
    print(f"Downloading historical data for {len(tickers)} tickers...")
    hist_data = yf.download(tickers, period="1y", interval="1d", progress=False)

    results = []

    for ticker in tickers:
        try:
            # Fetch Ticker object for P/E Ratio (Fundamental)
            t_obj = yf.Ticker(ticker)
            info = t_obj.info

            # Extract price series for this ticker
            ticker_hist = hist_data.xs(ticker, axis=1, level=1).dropna()

            if len(ticker_hist) < 2:
                continue

            close_prices = ticker_hist['Close']
            high_prices = ticker_hist['High']

            current_price = close_prices.iloc[-1]
            prev_close = close_prices.iloc[-2]

            # --- 1. NEW: Daily % Change (Current vs Last Closure) ---
            daily_pct_change = ((current_price - prev_close) / prev_close) * 100

            # --- 2. % Change (1 Month) ---
            price_1m_ago = close_prices.iloc[-22] if len(close_prices) >= 22 else close_prices.iloc[0]
            pct_change_1m = ((current_price - price_1m_ago) / price_1m_ago) * 100

            # --- 3. % Off 52-Week High ---
            high_52w = high_prices.max()
            pct_off_52w_high = ((current_price - high_52w) / high_52w) * 100

            # --- 4. RSI (14-Day) ---
            delta = close_prices.diff()
            gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
            loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()

            if loss.iloc[-1] == 0:
                rsi = 100 if gain.iloc[-1] > 0 else 50
            else:
                rs = gain / loss
                rsi = 100 - (100 / (1 + rs)).iloc[-1]

            # --- 5. Fundamental: P/E Ratio ---
            pe_ratio = info.get('trailingPE') or info.get('forwardPE')

            # --- 6. Sector ---
            sector = df_tickers[df_tickers.Ticker == ticker].Sector.iloc[0]

            # --- 7. Currency ---
            currency = df_tickers[df_tickers.Ticker == ticker].Currency.iloc[0]

            results.append({
                'Ticker': ticker,
                'Price': round(current_price, 2),
                '% 1D': f"{daily_pct_change:.1f}%",
                '% 1M': f"{pct_change_1m:.1f}%",
                '% 52W H': f"{pct_off_52w_high:.1f}%",
                'RSI': round(rsi, 1),
                'PE': round(pe_ratio, 1) if pe_ratio else "N/A",
                'Sector': sector,
                'Currency': currency
            })

        except Exception as e:
            print(f"Error processing {ticker}: {e}")

    # Create and save the final report
    df_final = pd.DataFrame(results)
    #df_final.to_csv('final_stock_report.csv', index=False)
    return df_final

### Fetch price history and calculate metrics

In [21]:
# Execute the update
analysis_results = get_complete_stock_analysis('dividend.csv')
print(analysis_results.head())

     Ticker   Price   % 1D  % 1M % 52W H   RSI    PE Sector Currency
0  NORCO.OL   41.50   1.8%  4.7%  -16.3%  68.7  19.5    IND      NOK
1   ATEA.OL  144.00   0.6%  1.8%  -12.1%  64.8  18.6    TEC      NOK
2    DNB.OL  306.20  -1.3%  5.3%   -2.4%  73.3  10.8    FIN      NOK
3   NONG.OL  160.56  -0.6%  6.8%   -1.7%  84.4  10.5    FIN      NOK
4    STB.OL  174.40   0.1%  4.2%   -1.9%  78.8  14.9    FIN      NOK


### Sort and format dataframe

In [22]:
df_sorted = analysis_results.sort_values(by = ["Sector", "PE"])
print(df_sorted)

      Ticker   Price   % 1D   % 1M % 52W H   RSI    PE Sector Currency
15  MORLD.OL   18.50   0.0%  10.5%   -5.8%  84.4  11.6    ENE      NOK
16    ODL.OL  102.40  -0.2%   2.4%   -6.6%  58.4  15.1    ENE      NOK
7     ENH.OL    8.57   0.2%  -4.4%  -16.4%  58.8  22.6    ENE      NOK
3    NONG.OL  160.56  -0.6%   6.8%   -1.7%  84.4  10.5    FIN      NOK
2     DNB.OL  306.20  -1.3%   5.3%   -2.4%  73.3  10.8    FIN      NOK
10   MING.OL  211.80   0.0%   7.6%   -1.2%  89.0  11.1    FIN      NOK
19      ITUB    9.33  -0.6%  15.7%   -1.9%  83.5  11.7    FIN      USD
11  SB1NO.OL  217.00  -0.9%   5.3%   -2.0%  75.8  12.8    FIN      NOK
9     B2I.OL   24.15  -1.4%  -3.2%   -5.7%  60.0  14.4    FIN      NOK
4     STB.OL  174.40   0.1%   4.2%   -1.9%  78.8  14.9    FIN      NOK
8     GJF.OL  264.80   0.5%   4.9%   -7.4%  83.9  21.1    FIN      NOK
13   MPCC.OL   22.78  -0.9%   6.4%   -5.4%  47.6   4.5    IND      NOK
12    WWI.OL  712.00  -1.0%  -2.1%   -9.1%  35.0   4.9    IND      NOK
14  HA

### Format HTML report and save

In [23]:
# Create the table HTML string
table_html = df_sorted.to_html(
    index=False, 
    classes='table table-striped table-hover table-sm', # Bootstrap classes
    border=0,
    justify='unset'
)

In [24]:
def color_change(val):
    color = 'green' if val > 0 else 'red'
    return f'color: {color}; font-weight: bold'

In [31]:
styled_html = (
    df_sorted.style
    .format({
        'Price': '{:.2f}',      # Force 2 decimals
        'PE': '{:.1f}',      # Force 1 decimal
        'RSI': '{:+.1f}'   # Force 1 decimal, a '+' sign for pos numbers, and a % symbol
    })
    .set_table_attributes('class="table table-striped table-hover"')
    .hide(axis='index')
    .to_html()
)

In [32]:
# Find local time
oslo_tz = pytz.timezone("Europe/Oslo")
oslo_time = pd.Timestamp.now(oslo_tz).strftime('%Y-%m-%d %H:%M')

# Inject styled_html into your template
html_template = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Sector Report</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <style>
        body {{ padding: 5px; background-color: #f8f9fa; }}
        .table-responsive {{ border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); background: white; margin: 0;}}
        h2 {{ color: #343a40; margin-bottom: 20px; font-weight: bold; }}
        /* Ensure the pandas-generated styles don't conflict with Bootstrap */
        table {{ width: 100%; }}
    </style>
</head>
<body>
    <div class="container">
        <h2>Sector Performance Report</h2>
        <div class="table-responsive">
            {styled_html}
        </div>
        <p class="text-muted mt-3">Last updated: {oslo_time}</p>
    </div>
</body>
</html>
"""

# Save the full report
with open("stock_report.html", "w") as f:
    f.write(html_template)